In [ ]:
import re
import sys
from pathlib import Path
import time

import pandas as pd


def update_inp_nodes(inp_text: str, mapping: dict[int, tuple[float, float, float]]) -> str:
    """
    Abaqus .inp の *NODE セクション内の
      node_id, x, y, z,
    という行の座標を mapping に基づいて置換する。

    - *NODE セクションは複数あっても全て対象
    - mapping に無い node_id は変更しない
    - * で始まる次のキーワード行が出たら *NODE セクション終了
    """

    lines = inp_text.splitlines()
    out_lines = []
    in_node = False

    for line in lines:
        stripped = line.strip()

        # *NODE 開始
        if stripped.upper().startswith("*NODE"):
            in_node = True
            out_lines.append(line)
            continue

        if in_node:
            # 次のキーワードが来たら *NODE 終了
            if stripped.startswith("*") and not stripped.upper().startswith("*NODE"):
                in_node = False
                out_lines.append(line)
                continue

            # 空行/コメントはそのまま
            if (not stripped) or stripped.startswith("**"):
                out_lines.append(line)
                continue

            # 節点行: 先頭の node_id を読む（例: "   8855713,  6.1E+02, ...")
            m = re.match(r"\s*(\d+)\s*,(.*)", line)
            if not m:
                out_lines.append(line)
                continue

            nid = int(m.group(1))
            if nid in mapping:
                x, y, z = mapping[nid]
                # 既存の見た目に近い scientific 表記で出力（例: 6.16782E+02）
                new_line = f"{nid:>10d},  {x:.5E},  {y:.5E},  {z:.5E},"
                out_lines.append(new_line)
            else:
                out_lines.append(line)
        else:
            out_lines.append(line)

    # 元ファイルが末尾改行ありなら維持
    return "\n".join(out_lines) + ("\n" if inp_text.endswith("\n") else "")


def main():
    t0 = time.perf_counter()
    # 入力ファイル
    csv_path = Path("../output/output.csv")
    # inp_path = Path("../inp/test.inp")
    inp_path = Path("../inp/Case_2nd_order_nlgeom.inp")

    # 出力ファイル
    out_inp_path = Path("../inp/test_updated.inp")

    # CSV読み込み（必要なら encoding を変更）
    df = pd.read_csv(csv_path)
    t1 = time.perf_counter()
    print(f"read csv: {t1 - t0:.3f} 秒")

    # Node Label → (X_new, Y_new, Z_new)
    mapping = {
        int(r["Node Label"]): (float(r["X_new"]), float(r["Y_new"]), float(r["Z_new"]))
        for _, r in df.iterrows()
    }

    # inp読み込み
    inp_text = inp_path.read_text(encoding="utf-8", errors="replace")
    t2 = time.perf_counter()
    print(f"read inp : {t2 - t1:.3f} 秒")

    # 更新
    updated_text = update_inp_nodes(inp_text, mapping)
    t3 = time.perf_counter()
    print(f"updated: {t3 - t2:.3f} 秒")

    # 保存
    out_inp_path.write_text(updated_text, encoding="utf-8")
    print(f"Updated INP saved: {out_inp_path.resolve()}")
    print(f"合計: {t3 - t0:.3f} 秒")


if __name__ == "__main__":
    main()


read csv: 5.060 秒
read inp : 151.027 秒
書き込み: 27.939 秒
Updated INP saved: C:\Users\z0114adm\Downloads\toyama_csv\inp\test_updated.inp
合計: 184.026 秒
